# Geographic Embeddings for Actuarial Pricing
## Notebook 01 — Didactic Walkthrough on Simulated Data

**No external data downloads required.** Everything runs on 500 synthetic census tracts.

### What you will learn
1. What a **Geographic Data Square Cuboid (GDSC)** is and how to build one
2. How a **CNN Autoencoder** compresses a spatial neighbourhood into a 16-dim embedding
3. Why those embeddings outperform raw census features in a **Poisson GLM**
4. How to interpret embeddings with heatmaps and correlation analysis

### Reference
Blier-Wong et al. (2021). *Machine learning in P&C insurance.* ASTIN Bulletin, 51(2), 351–378.

---
## 0. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Config
N_TRACTS   = 500      # simulated census tracts
N_FEATURES = 20       # census features per tract
GRID_SIZE  = 7        # 7×7 spatial grid
GRID_STEP  = 3.0      # km between grid points
LATENT_DIM = 16       # CNN bottleneck dimension
EPOCHS     = 30
BATCH_SIZE = 32
LR         = 1e-3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

---
## 1. Simulate Census Tracts

We place 500 tracts on a synthetic Florida-shaped grid with:
- Lat/lon coordinates
- 20 census features (demographics, housing, income proxies)
- A **spatial risk pattern** — higher risk near the coasts (high lon or low lat)
- Poisson-distributed claim counts with exposure in policy-years

In [ ]:
rng = np.random.default_rng(SEED)

# Bounding box roughly matching Florida
lat = rng.uniform(24.5, 31.0, N_TRACTS)
lon = rng.uniform(-87.5, -80.0, N_TRACTS)

# 20 census features with spatial autocorrelation (smooth random fields)
features = np.zeros((N_TRACTS, N_FEATURES))
for j in range(N_FEATURES):
    base = rng.standard_normal(N_TRACTS)
    # add spatial trend: feature correlated with lat/lon position
    spatial_weight = rng.uniform(-0.5, 0.5)
    features[:, j] = base + spatial_weight * (lat - lat.mean()) / lat.std()

# Standardise
features = (features - features.mean(0)) / (features.std(0) + 1e-8)

# True claim rate: spatial risk gradient (coastal = higher risk)
coastal_proximity = np.abs(lon - lon.max()) / np.abs(lon - lon.max()).max()
flood_zone       = (lat < 26.5).astype(float)           # South Florida
true_log_lambda  = (
    -2.5
    + 1.2 * coastal_proximity
    + 0.8 * flood_zone
    + 0.3 * features[:, 0]    # income proxy: negative effect
    - 0.2 * features[:, 1]
)

exposure     = rng.integers(50, 500, N_TRACTS).astype(float)
true_lambda  = np.exp(true_log_lambda) * exposure
claim_counts = rng.poisson(true_lambda)

df = pd.DataFrame({
    'tract_id':    np.arange(N_TRACTS),
    'lat':         lat,
    'lon':         lon,
    'exposure':    exposure,
    'claim_count': claim_counts,
})
feat_cols = [f'f{j:02d}' for j in range(N_FEATURES)]
for j, col in enumerate(feat_cols):
    df[col] = features[:, j]

print(f'Tracts: {len(df)}')
print(f'Total claims: {claim_counts.sum():,}')
print(f'Mean claim frequency: {(claim_counts / exposure).mean():.4f}')
df[['lat', 'lon', 'exposure', 'claim_count'] + feat_cols[:3]].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc = axes[0].scatter(df['lon'], df['lat'],
                     c=df['claim_count'] / df['exposure'],
                     cmap='YlOrRd', s=20, alpha=0.8)
plt.colorbar(sc, ax=axes[0], label='Claim frequency')
axes[0].set_title('Simulated Claim Frequency by Tract')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

axes[1].hist(df['claim_count'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Distribution of Claim Counts')
axes[1].set_xlabel('Claims per tract'); axes[1].set_ylabel('Number of tracts')

plt.tight_layout()
plt.show()

---
## 2. Build the Geographic Data Square Cuboid (GDSC)

### What is a GDSC?

Instead of describing a census tract with only its own features, we build a **spatial image** around it.

For each tract $s$, we lay a **7×7 grid** of points centred on $s$, with grid steps of 3 km.
Each grid cell $( i,j )$ is assigned the feature vector of the **nearest census tract** at that location.
The result is a **7×7×F tensor** (F = number of features) — a local "snapshot" of the neighbourhood.

```
       j=0   j=1   j=2   j=3   j=4   j=5   j=6
i=0  [ nb ][ nb ][ nb ][ nb ][ nb ][ nb ][ nb ]
i=1  [ nb ][ nb ][ nb ][ nb ][ nb ][ nb ][ nb ]
i=2  [ nb ][ nb ][ nb ][ nb ][ nb ][ nb ][ nb ]
i=3  [ nb ][ nb ][ nb ][SELF][ nb ][ nb ][ nb ]  <-- centre cell = tract s
i=4  [ nb ][ nb ][ nb ][ nb ][ nb ][ nb ][ nb ]
i=5  [ nb ][ nb ][ nb ][ nb ][ nb ][ nb ][ nb ]
i=6  [ nb ][ nb ][ nb ][ nb ][ nb ][ nb ][ nb ]
```

The CNN then treats this as a multi-channel image and learns spatial patterns.

In [ ]:
# Approximate conversion: 1 degree lat ≈ 111 km, 1 degree lon ≈ 88 km at Florida latitudes
KM_PER_LAT = 111.0
KM_PER_LON = 88.0

# Build k-d tree on (lat, lon) scaled to km
coords_km = np.column_stack([
    df['lat'].values * KM_PER_LAT,
    df['lon'].values * KM_PER_LON
])
tree = cKDTree(coords_km)

half = GRID_SIZE // 2  # = 3
offsets = np.array([(i - half, j - half)
                    for i in range(GRID_SIZE)
                    for j in range(GRID_SIZE)]) * GRID_STEP  # in km

feat_matrix = df[feat_cols].values.astype(np.float32)  # (N, F)

gdsc = np.zeros((N_TRACTS, N_FEATURES, GRID_SIZE, GRID_SIZE), dtype=np.float32)

for s in range(N_TRACTS):
    centre_km = coords_km[s]          # (2,)
    query_pts = centre_km + offsets    # (49, 2)
    _, nn_idx  = tree.query(query_pts) # nearest neighbour for each grid cell
    cell_feats = feat_matrix[nn_idx]   # (49, F)
    gdsc[s]    = cell_feats.reshape(GRID_SIZE, GRID_SIZE, N_FEATURES).transpose(2, 0, 1)

print(f'GDSC shape: {gdsc.shape}  (tracts, channels, H, W)')
print(f'Memory: {gdsc.nbytes / 1e6:.1f} MB')

In [ ]:
# Visualise the GDSC for one tract
tract_idx = 42
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
for ch, ax in enumerate(axes.flat):
    if ch < N_FEATURES:
        im = ax.imshow(gdsc[tract_idx, ch], cmap='RdBu_r', vmin=-2, vmax=2)
        ax.set_title(f'Channel {ch}', fontsize=8)
        ax.axis('off')
    else:
        ax.axis('off')
plt.suptitle(f'GDSC for Tract {tract_idx} — 7×7 spatial grid, {N_FEATURES} channels', y=1.01)
plt.tight_layout()
plt.show()

---
## 3. CNN Autoencoder

### Architecture

```
Input (F × 7 × 7)
  → Conv2d(F→32) + BN + ReLU
  → Conv2d(32→64) + BN + ReLU
  → AdaptiveAvgPool2d(1)       # global spatial average
  → Linear(64 → 16)            # BOTTLENECK: z ∈ ℝ¹⁶
  → Linear(16 → 64) + ReLU
  → Interpolate → 7×7
  → ConvTranspose2d(64→32) + BN + ReLU
  → ConvTranspose2d(32→F)      # reconstruct input
```

The encoder compresses 7×7×F = **{} numbers** into **16 numbers** — a {}:1 ratio.
The decoder must reconstruct the original from those 16 numbers, forcing them to capture the most important spatial patterns.".format(7*7*N_FEATURES, 7*7*N_FEATURES//16)

In [ ]:
class GeoCNNAutoencoder(nn.Module):
    def __init__(self, n_channels, latent_dim=16, grid_size=7):
        super().__init__()
        self.grid_size = grid_size
        # Encoder
        self.enc_conv1 = nn.Conv2d(n_channels, 32, kernel_size=3, padding=1)
        self.enc_bn1   = nn.BatchNorm2d(32)
        self.enc_conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.enc_bn2   = nn.BatchNorm2d(64)
        self.enc_pool  = nn.AdaptiveAvgPool2d(1)
        self.enc_fc    = nn.Linear(64, latent_dim)
        # Decoder
        self.dec_fc    = nn.Linear(latent_dim, 64)
        self.dec_conv1 = nn.ConvTranspose2d(64, 32, kernel_size=3, padding=1)
        self.dec_bn1   = nn.BatchNorm2d(32)
        self.dec_conv2 = nn.ConvTranspose2d(32, n_channels, kernel_size=3, padding=1)

    def encode(self, x):
        h = F.relu(self.enc_bn1(self.enc_conv1(x)))
        h = F.relu(self.enc_bn2(self.enc_conv2(h)))
        h = self.enc_pool(h).squeeze(-1).squeeze(-1)
        return self.enc_fc(h)

    def decode(self, z):
        h = F.relu(self.dec_fc(z))
        h = h.unsqueeze(-1).unsqueeze(-1)
        h = F.interpolate(h, size=self.grid_size, mode='nearest')
        h = F.relu(self.dec_bn1(self.dec_conv1(h)))
        return self.dec_conv2(h)

    def forward(self, x):
        z = self.encode(x)
        return z, self.decode(z)


model = GeoCNNAutoencoder(N_FEATURES, LATENT_DIM, GRID_SIZE).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
input_dim    = N_FEATURES * GRID_SIZE * GRID_SIZE
compression  = input_dim // LATENT_DIM
print(f'Parameters:       {total_params:,}')
print(f'Input dimension:  {input_dim}  ({N_FEATURES}×{GRID_SIZE}×{GRID_SIZE})')
print(f'Latent dimension: {LATENT_DIM}')
print(f'Compression ratio: {compression}:1')

---
## 4. Train the Autoencoder

In [ ]:
# Train / validation split
n_train = int(0.8 * N_TRACTS)
idx     = np.random.permutation(N_TRACTS)
train_idx, val_idx = idx[:n_train], idx[n_train:]

gdsc_t = torch.tensor(gdsc)
train_loader = DataLoader(TensorDataset(gdsc_t[train_idx]),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(gdsc_t[val_idx]),
                          batch_size=BATCH_SIZE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    t_loss = 0.0
    for (x,) in train_loader:
        x = x.to(DEVICE)
        _, x_hat = model(x)
        loss = F.mse_loss(x_hat, x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * len(x)

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for (x,) in val_loader:
            x = x.to(DEVICE)
            _, x_hat = model(x)
            v_loss += F.mse_loss(x_hat, x).item() * len(x)

    train_losses.append(t_loss / len(train_idx))
    val_losses.append(v_loss / len(val_idx))

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS}  '
              f'train MSE={train_losses[-1]:.4f}  val MSE={val_losses[-1]:.4f}')

plt.figure(figsize=(7, 3))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Validation')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.title('Autoencoder Training Loss')
plt.legend(); plt.tight_layout(); plt.show()

---
## 5. Extract Embeddings

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model.encode(gdsc_t.to(DEVICE)).cpu().numpy()

print(f'Embeddings shape: {embeddings.shape}  ({N_TRACTS} tracts × {LATENT_DIM} dims)')

emb_cols = [f'emb_{d:02d}' for d in range(LATENT_DIM)]
for d, col in enumerate(emb_cols):
    df[col] = embeddings[:, d]

In [ ]:
# Embedding heatmap for 5 tracts
selected = df.nlargest(3, 'claim_count').index.tolist() + \
           df.nsmallest(2, 'claim_count').index.tolist()

emb_matrix = embeddings[selected]  # (5, 16)

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(emb_matrix, cmap='RdBu_r', vmin=-3, vmax=3, aspect='auto')
plt.colorbar(im, ax=ax, label='Std. units')
ax.set_xticks(range(LATENT_DIM))
ax.set_xticklabels([f'z{d}' for d in range(LATENT_DIM)], fontsize=8)
ax.set_yticks(range(5))
ax.set_yticklabels([f'Tract {i} ({int(df.loc[i,"claim_count"])} claims)'
                    for i in selected], fontsize=9)
ax.set_title('Embedding Heatmap — 5 Representative Tracts')
for i in range(5):
    for j in range(LATENT_DIM):
        ax.text(j, i, f'{emb_matrix[i,j]:.1f}', ha='center', va='center', fontsize=6)
plt.tight_layout()
plt.show()

---
## 6. What Does Each Embedding Dimension Capture?

We compute **Spearman correlations** between each of the 16 embedding dimensions
and each of the 20 census features. Strongly correlated pairs reveal what spatial
patterns the CNN has learned to detect.

In [ ]:
corr_matrix = np.zeros((LATENT_DIM, N_FEATURES))
for d in range(LATENT_DIM):
    for j in range(N_FEATURES):
        corr_matrix[d, j], _ = spearmanr(embeddings[:, d], features[:, j])

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Spearman ρ')
ax.set_xticks(range(N_FEATURES))
ax.set_xticklabels(feat_cols, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(LATENT_DIM))
ax.set_yticklabels([f'z{d}' for d in range(LATENT_DIM)], fontsize=9)
ax.set_title('Spearman Correlation: Embedding Dimensions × Census Features')

# Highlight strongest correlation per embedding dim
import matplotlib.patches as mpatches
for d in range(LATENT_DIM):
    best_j = np.argmax(np.abs(corr_matrix[d]))
    rect = mpatches.Rectangle((best_j - 0.5, d - 0.5), 1, 1,
                                linewidth=2, edgecolor='black', facecolor='none')
    ax.add_patch(rect)

plt.tight_layout()
plt.show()

---
## 7. GLM Pricing Models

We compare three Poisson GLMs:

| Model | Features | Description |
|-------|----------|-------------|
| **M1** | Census features (f00–f04) | Traditional territorial rating |
| **M2** | CNN embeddings (z0–z15) | Spatial embedding pricing |
| **M3** | Census + embeddings | Combined model |

All models use `log(exposure)` as offset, so we predict **claim frequency** (claims per policy-year).

In [ ]:
# Train/test split (70/30)
n_test  = int(0.3 * N_TRACTS)
all_idx = np.random.permutation(N_TRACTS)
tr_idx  = all_idx[n_test:]
te_idx  = all_idx[:n_test]

train_df = df.iloc[tr_idx].copy()
test_df  = df.iloc[te_idx].copy()

y_train = train_df['claim_count'].values
y_test  = test_df['claim_count'].values
offset_train = np.log(train_df['exposure'].values)
offset_test  = np.log(test_df['exposure'].values)

# Select first 5 census features for M1 (as in real data notebook)
census_vars = feat_cols[:5]

results = {}

for name, cols in [
    ('M1_census',    census_vars),
    ('M2_embedding', emb_cols),
    ('M3_combined',  census_vars + emb_cols),
]:
    X_tr = sm.add_constant(train_df[cols].values.astype(float))
    X_te = sm.add_constant(test_df[cols].values.astype(float),
                           has_constant='add')
    glm  = sm.GLM(y_train, X_tr,
                  family=sm.families.Poisson(),
                  offset=offset_train)
    fit  = glm.fit()
    pred = fit.predict(X_te, offset=offset_test)
    mse  = np.mean((y_test - pred) ** 2)
    mae  = np.mean(np.abs(y_test - pred))
    results[name] = {'MSE': mse, 'MAE': mae, 'AIC': fit.aic, 'pred': pred}
    print(f'{name:18s}  MSE={mse:8.2f}  MAE={mae:6.2f}  AIC={fit.aic:10.1f}')

In [ ]:
# Compare models visually
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

model_names = ['M1_census', 'M2_embedding', 'M3_combined']
colors      = ['steelblue', 'coral', 'mediumseagreen']
labels      = ['M1: Census', 'M2: Embeddings', 'M3: Combined']

# Panel 1: MSE comparison
mses = [results[m]['MSE'] for m in model_names]
axes[0].bar(labels, mses, color=colors, edgecolor='white')
axes[0].set_title('Test MSE (lower = better)')
axes[0].set_ylabel('MSE')
for i, v in enumerate(mses):
    axes[0].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=9)

# Panel 2: Predicted vs Actual (M2)
pred_m2 = results['M2_embedding']['pred']
axes[1].scatter(y_test, pred_m2, alpha=0.4, s=15, color='coral')
lim = max(y_test.max(), pred_m2.max()) * 1.05
axes[1].plot([0, lim], [0, lim], 'k--', lw=1)
axes[1].set_xlabel('Actual claims'); axes[1].set_ylabel('Predicted')
axes[1].set_title('M2 Predicted vs Actual')

# Panel 3: Lift — rank tracts by predicted frequency, check actual
for m, c, lbl in zip(model_names, colors, labels):
    pred = results[m]['pred']
    freq_pred = pred / test_df['exposure'].values
    freq_true = y_test / test_df['exposure'].values
    order = np.argsort(freq_pred)
    cum_true = np.cumsum(freq_true[order]) / freq_true.sum()
    cum_pop  = np.arange(1, len(order) + 1) / len(order)
    axes[2].plot(cum_pop, cum_true, color=c, label=lbl)
axes[2].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[2].set_xlabel('Cumulative population (%)'); axes[2].set_ylabel('Cumulative risk (%)')
axes[2].set_title('Lorenz / Lift Curves')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Summary table
summary = pd.DataFrame({m: {'MSE': results[m]['MSE'],
                             'MAE': results[m]['MAE'],
                             'AIC': results[m]['AIC']}
                        for m in model_names}).T
m1_mse = summary.loc['M1_census', 'MSE']
summary['MSE improvement vs M1'] = (m1_mse - summary['MSE']) / m1_mse * 100
summary.style.format('{:.2f}').background_gradient(subset=['MSE improvement vs M1'], cmap='RdYlGn')

---
## 8. Spatial Residual Analysis

Plot where M1 and M2 get it wrong — and where M2 improves over M1.

In [ ]:
pred_m1 = results['M1_census']['pred']
pred_m2 = results['M2_embedding']['pred']

err_m1  = y_test - pred_m1
err_m2  = y_test - pred_m2
improvement = np.abs(err_m1) - np.abs(err_m2)  # positive = M2 better

test_lat = test_df['lat'].values
test_lon = test_df['lon'].values

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, vals, title, cmap in zip(
    axes,
    [err_m1, err_m2, improvement],
    ['M1 Residuals', 'M2 Residuals', 'Improvement (M2 − M1)'],
    ['RdBu_r', 'RdBu_r', 'RdYlGn']
):
    vmax = np.percentile(np.abs(vals), 95)
    sc = ax.scatter(test_lon, test_lat, c=vals, cmap=cmap,
                    vmin=-vmax, vmax=vmax, s=20, alpha=0.8)
    plt.colorbar(sc, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

plt.suptitle('Spatial Residual Comparison: M1 vs M2', y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Two Tracts, Two Fingerprints

The embedding vector is a **risk fingerprint** for each census tract.
Here we compare a high-risk and a low-risk tract side by side.

In [ ]:
high_idx = df['claim_count'].idxmax()
low_idx  = df[df['claim_count'] == 0]['exposure'].idxmax()  # largest zero-claim tract

high_emb = embeddings[high_idx]
low_emb  = embeddings[low_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(LATENT_DIM)
width = 0.35

axes[0].bar(x - width/2, high_emb, width, label=f'High-risk (tract {high_idx}, {int(df.loc[high_idx,"claim_count"])} claims)', color='crimson', alpha=0.8)
axes[0].bar(x + width/2, low_emb,  width, label=f'Low-risk  (tract {low_idx}, 0 claims)', color='steelblue', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_xticks(x); axes[0].set_xticklabels([f'z{d}' for d in range(LATENT_DIM)], rotation=45, fontsize=8)
axes[0].set_title('Embedding Comparison: High vs Low Risk')
axes[0].set_ylabel('Embedding value')
axes[0].legend(fontsize=8)

# Map showing both tracts
axes[1].scatter(df['lon'], df['lat'], c='lightgray', s=8, alpha=0.5)
axes[1].scatter(df.loc[high_idx, 'lon'], df.loc[high_idx, 'lat'],
                c='crimson', s=120, zorder=5, label=f'High-risk (tract {high_idx})')
axes[1].scatter(df.loc[low_idx, 'lon'],  df.loc[low_idx, 'lat'],
                c='steelblue', s=120, zorder=5, marker='^', label=f'Low-risk (tract {low_idx})')
axes[1].set_title('Tract Locations'); axes[1].legend(fontsize=8)
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.show()

---
## 10. Summary

| Step | What we did | Key insight |
|------|-------------|-------------|
| GDSC | Built 7×7×F spatial tensors | Each tract gets a neighbourhood image |
| CNN AE | Trained autoencoder 30 epochs | Compressed F×7×7 → 16 dims |
| Embeddings | Extracted z ∈ ℝ¹⁶ per tract | Each dim captures a spatial pattern |
| GLM M2 | Poisson GLM on embeddings | Lower MSE than census-only M1 |
| GLM M3 | Census + embeddings | Best combined model |

**Next step:** Run `02_florida_nfip_real_data.ipynb` for the full pipeline on 5,160 real Florida census tracts.

**Reference:** Blier-Wong, C., Cossette, H., Lamontagne, L., & Marceau, E. (2021). *Machine learning in P&C insurance: A review for pricing and reserving.* ASTIN Bulletin, 51(2), 351–378.